# ETL - Extract, Transform, Load

This notebook covers two stages:

1. **Raw Data Audit** - profile every source file, surface quality issues, and document transformation decisions as inline notes.
2. **Transformation** - call `scripts/transform.py` to build clean analysis tables and export them to CSV, SQLite, and Excel.

> **Run `01_acquire_data.ipynb` first** to populate `data/raw/` before running this notebook.

In [1]:
import sys
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebook":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
from scripts.config import RAW, CLEAN
from scripts.transform import (
    build_socioeconomic,
    build_population,
    build_health,
    build_combined,
    build_cpi,
    build_sdi,
    run,
 )

---
## 1. Raw Data Audit

Each dataset is profiled for shape, column types, missing values, key uniqueness, and year/state coverage. Issues discovered here informed the ETL design choices documented below.

In [ ]:
FILES = {
    "income_parlimen":  RAW / "hh_income_parlimen.csv",
    "poverty_parlimen": RAW / "hh_poverty_parlimen.csv",
    "income_hist":      RAW / "hh_income_state.csv",
    "poverty_hist":     RAW / "hh_poverty_state.csv",
    "inequality":       RAW / "hh_inequality_state.csv",
    "population":       RAW / "population_state.csv",
    "cpi_national":     RAW / "cpi_national.csv",
    "facilities":       RAW / "moh_facilities.csv",
    "beds":             RAW / "moh_beds.csv",
    "worldbank":        RAW / "worldbank_malaysia.csv",
}

dfs = {name: pd.read_csv(path) for name, path in FILES.items()}

print(f"{'Dataset':<22} {'Rows':>6}  {'Cols':>4}  {'File':}")
print("-" * 70)
for name, df in dfs.items():
    print(f"{name:<22} {len(df):>6}  {df.shape[1]:>4}  {FILES[name].name}")

### 1.1 Schema & State Coverage
Check that all 16 states/territories appear in each file after normalising name variants.

In [ ]:
from scripts.config import STATE_NORM

EXPECTED_STATES = {
    "Johor", "Kedah", "Kelantan", "Melaka", "Negeri Sembilan", "Pahang",
    "Perak", "Perlis", "Pulau Pinang", "Sabah", "Sarawak", "Selangor",
    "Terengganu", "W.P. Kuala Lumpur", "W.P. Labuan", "W.P. Putrajaya",
}

def norm_states(df, col="state"):
    return set(df[col].astype(str).str.strip().replace(STATE_NORM).dropna().unique())

print(f"{'Dataset':<22} {'Count':>5}  Missing states")
print("-" * 70)
for name in ["income_parlimen", "poverty_parlimen", "facilities", "beds",
             "population", "income_hist", "poverty_hist", "inequality"]:
    states = norm_states(dfs[name])
    missing = sorted(EXPECTED_STATES - states)
    print(f"{name:<22} {len(states):>5}  {missing if missing else 'none'}")

# National CPI has no state dimension (single national series)
cpi_years = sorted(pd.to_datetime(dfs["cpi_national"]["date"]).dt.year.unique().astype(int))
print(f"{'cpi_national':<22}       national series, {len(cpi_years)} years: {cpi_years[0]}–{cpi_years[-1]}")

### 1.2 Year Coverage

Check year coverage of raw datasets

In [ ]:
for name in ["income_parlimen", "poverty_parlimen", "income_hist", "poverty_hist",
             "population", "inequality"]:
    years = sorted(
        pd.to_datetime(dfs[name]["date"], errors="coerce").dt.year.dropna().unique().astype(int)
    )
    print(f"{name:<22} {years}")

# National CPI year coverage (annual — one row per division per year)
cpi_years = sorted(pd.to_datetime(dfs["cpi_national"]["date"]).dt.year.unique().astype(int))
n_rows = len(dfs["cpi_national"])
print(f"{'cpi_national':<22} {cpi_years[0]}–{cpi_years[-1]}  (annual; {n_rows} rows × {len(cpi_years)} years × {n_rows // len(cpi_years)} divisions)")

### 1.3 Missing Values & Duplicates

Check for missing values and duplicates

In [ ]:
def quality_report(name, df, key_cols=None):
    print(f"\n=== {name.upper()} ===")
    missing = df.isna().sum()
    missing = missing[missing > 0].sort_values(ascending=False)
    print("Missing:", missing.to_string() if not missing.empty else "  none")
    print("Full-row duplicates:", df.duplicated().sum())
    if key_cols:
        print(f"Key duplicates {key_cols}:", df.duplicated(subset=key_cols).sum())

quality_report("income_parlimen",  dfs["income_parlimen"],  ["date", "state", "parlimen"])
quality_report("poverty_parlimen", dfs["poverty_parlimen"], ["date", "state", "parlimen"])
quality_report("beds",             dfs["beds"],             ["hospital", "state"])
quality_report("facilities",       dfs["facilities"],       ["name", "state"])
quality_report("population",       dfs["population"],       ["date", "state", "age", "sex", "ethnicity"])
quality_report("income_hist",      dfs["income_hist"],      ["date", "state"])
quality_report("poverty_hist",     dfs["poverty_hist"],     ["date", "state"])
quality_report("inequality",       dfs["inequality"],       ["date", "state"])
quality_report("cpi_national",     dfs["cpi_national"],     ["date", "division"])

### 1.4 Data Quality Notes & ETL Decisions

The following issues were found during exploration. Each is addressed in the transformation step below.

| Issue | Dataset | Decision |
|---|---|---|
| State name variants (`W.P Kuala Lumpur`, `WP Labuan`, etc.) | All files | Normalise via `STATE_NORM` mapping before any aggregation |
| `beds_nonicu` has one record with value `-4` (data entry error) | `moh_beds` | Clamp negative values to 0 with `clip(lower=0)` |
| `poverty_hardcore` and `poverty_relative` absent before 1989 | `hh_poverty_state` | Drop these columns from the historical combine; only `poverty_absolute` is consistent |
| Uneven state counts in early survey years (11-14 states in 1970s) | `hh_income_state` | Include as-is; NaN naturally fills missing state-years in the combined table |
| Population API filter returned wrong column names | `population_state.csv` | Download direct CSV URL instead of using the API filter endpoint |
| `hh_income_parlimen` and `hh_poverty_parlimen` keys are perfectly aligned | Both | Inner merge on `[state, year]` after aggregation — no rows lost |
| Parliament-level Gini is inter-constituency dispersion, not household | Derived | Replaced by official DOSM household Gini from `hh_inequality_state` where available |
| `hh_inequality_state` covers only survey years (not every year) | `hh_inequality_state` | Left-joined to `combined_state`; non-survey years remain NaN |
| State CPI only covered 2021–2023; most historical rows would be null | `cpi_state` (dropped) | Replaced with national annual CPI (`cpi_2d_annual`, 1960–present); joined to `combined_state` on `year` — same deflator for all states |
| CPI base year is 2010=100; HIES income figures are nominal | Both | `cpi_overall` attached to `combined_state` for all survey years with data overlap |

---
## 2. Transformation

All build functions live in `scripts/transform.py` and are called here. Each produces a labelled DataFrame which is then displayed for inspection before the final export.

### 1.1 Socioeconomic table

In [7]:
socio = build_socioeconomic()
print(f"Shape: {socio.shape}  |  States x years: {socio.groupby('year')['state'].count().to_dict()}")
display(socio.head(6))
display(socio.describe())

Shape: (48, 9)  |  States x years: {2019: 16, 2022: 16, 2024: 16}


,state_code,state,year,territory_type,income_mean,income_median,poverty_absolute,gini,parlimen_count
0,JHR,Johor,2019,state,7201.307692,5882.884615,4.623077,0.107621,26
1,JHR,Johor,2022,state,7529.730769,6215.346154,5.630769,0.108918,26
2,JHR,Johor,2024,state,8398.769231,7112.923077,3.092308,0.120739,26
3,KDH,Kedah,2019,state,5407.800000,4309.400000,8.540000,0.075098,15
4,KDH,Kedah,2022,state,5442.600000,4349.266667,8.346667,0.075074,15
5,KDH,Kedah,2024,state,5692.666667,4757.733333,8.133333,0.059273,15


,year,income_mean,income_median,poverty_absolute,gini,parlimen_count
count,48.000000,48.000000,48.000000,48.000000,48.000000,48.000000
mean,2021.666667,7639.813824,6096.484513,6.191730,0.080421,13.875000
std,2.076549,2733.370468,2139.265724,5.316206,0.043265,9.192099
min,2019.000000,4792.928571,3566.071429,0.100000,0.000000,1.000000
25%,2019.000000,5529.238750,4492.596250,2.111538,0.056967,7.500000
50%,2022.000000,6578.937500,5303.312500,4.774038,0.080981,13.500000
75%,2024.000000,8429.076923,6956.230769,8.351667,0.117108,22.500000
max,2024.000000,14028.636364,10909.636364,21.824000,0.151553,31.000000


### 1.2 Population table

In [8]:
print("--- Population table ---")
pop = build_population()
print(f"Shape: {pop.shape}  |  Years: {sorted(pop['year'].unique())}")
display(pop.pivot(index='state', columns='year', values='population').round(0))

--- Population table ---
Shape: (48, 3)  |  Years: [np.int32(2019), np.int32(2022), np.int32(2024)]


year,2019,2022,2024
state,,,
Johor,3761200.0,4028300.0,4184400.0
Kedah,2173700.0,2163100.0,2217100.0
Kelantan,1883800.0,1830600.0,1887900.0
Melaka,928400.0,1008600.0,1046700.0
Negeri Sembilan,1126200.0,1207900.0,1239500.0
Pahang,1671400.0,1614300.0,1667700.0
Perak,2508800.0,2514400.0,2569400.0
Perlis,254000.0,289800.0,296800.0
Pulau Pinang,1768800.0,1740900.0,1800500.0


### 1.3 Health Infrastructure table

In [9]:
health = build_health(pop)
print(f"Shape: {health.shape}")
display(health[health.year == 2022][
    ['state', 'beds_per_1000', 'facilities_per_100k', 'hospital_count']
].sort_values('beds_per_1000', ascending=False))

Shape: (48, 12)


,state,beds_per_1000,facilities_per_100k,hospital_count
46,W.P. Putrajaya,6.752137,5.128205,2
22,Perlis,1.766736,14.147688,1
31,Sarawak,1.526177,11.885992,23
19,Perak,1.389198,14.556157,16
7,Kelantan,1.367858,14.803889,10
13,Negeri Sembilan,1.358556,14.074013,8
37,Terengganu,1.324794,16.517782,6
10,Melaka,1.315685,11.104501,4
4,Kedah,1.278720,14.377514,9
43,W.P. Labuan,1.259030,14.447884,1


### 1.4 Combined historical table (1970-2024)

In [10]:
combined = build_combined(socio)
print(f"Shape: {combined.shape}  |  Years: {combined['year'].min()}-{combined['year'].max()}")
gini_coverage = combined['gini'].notna().sum()
print(f"Gini populated: {gini_coverage}/{len(combined)} rows  "
      f"({100*gini_coverage/len(combined):.0f}%  — official DOSM where available, NaN elsewhere)")
display(combined[combined['gini'].notna()].head(6))

Shape: (319, 8)  |  Years: 1970-2024
Gini populated: 289/319 rows  (91%  — official DOSM where available, NaN elsewhere)


,state_code,state,year,territory_type,income_mean,income_median,poverty_absolute,gini
1,JHR,Johor,1974,state,382.0,269.0,NaN,0.439
2,JHR,Johor,1976,state,513.0,370.0,29.0,0.469
3,JHR,Johor,1979,state,731.0,518.0,18.2,0.442
4,JHR,Johor,1984,state,1065.0,793.0,12.2,0.404
5,JHR,Johor,1987,state,1060.0,816.0,11.1,0.386
6,JHR,Johor,1989,state,1150.0,880.0,10.1,0.381


### 1.5 CPI table

In [ ]:
cpi = build_cpi()
print(f"Shape: {cpi.shape}  |  Years: {cpi['year'].min()}–{cpi['year'].max()}")
display(cpi.tail(10))

### 1.6 Socioeconomic Deprivation Index

In [12]:
sdi = build_sdi(socio, health)
print(f"Shape: {sdi.shape}")
display(sdi[['state', 'sdi_score', 'sdi_rank', 'double_deprivation']])

Shape: (16, 19)


,state,sdi_score,sdi_rank,double_deprivation
0,Sabah,0.883372,1,True
1,Sarawak,0.786600,2,True
2,Kelantan,0.693774,3,False
3,Negeri Sembilan,0.630413,4,True
4,Perak,0.616791,5,False
5,Kedah,0.608735,6,False
6,Johor,0.595962,7,True
7,Pulau Pinang,0.555300,8,False
8,Pahang,0.532351,9,False
9,Selangor,0.511774,10,False


---
## 3. Export

`run()` orchestrates all build steps above and saves to:
- `data/clean/combined_state.csv` — historical socioeconomic series with official Gini and national CPI attached
- `data/clean/health_state.csv`
- `data/clean/sdi_scores.csv`
- `data/clean/cpi_national.csv` — national annual headline CPI (1960–present, base 2010=100)
- `malaysia_project.db` (SQLite — 4 tables)
- `data/clean/dashboard_data.xlsx` (4 sheets)

In [13]:
run()


--- Building socioeconomic table ---
  socioeconomic: (48, 9)

--- Building population table ---
  population: (48, 3)

--- Building health infrastructure table ---
  health: (48, 12)

--- Building combined historical table ---
  combined: (319, 8), years 1970–2024

--- Building CPI table ---
  cpi: (42, 4), years 2021–2023

--- Computing SDI scores ---
  sdi: (16, 19)

--- Saving outputs ---
  ✓ combined_state.csv
  ✓ health_state.csv
  ✓ sdi_scores.csv
  ✓ cpi_state.csv
  ✓ malaysia_project.db (tables: combined_state, health_state, sdi_scores, cpi_state)
✓ dashboard_data.xlsx: 4 sheets (sdi_scores, combined_state, health_state, cpi_state)


### Check output files

In [ ]:
OUTPUTS = {
    "combined_state.csv": CLEAN / "combined_state.csv",
    "health_state.csv":   CLEAN / "health_state.csv",
    "sdi_scores.csv":     CLEAN / "sdi_scores.csv",
    "cpi_national.csv":   CLEAN / "cpi_national.csv",
}

print(f"{'File':<25} {'Rows':>6}  Size")
print("-" * 50)
for label, path in OUTPUTS.items():
    if path.exists():
        df = pd.read_csv(path)
        size = path.stat().st_size
        print(f"{label:<25} {len(df):>6}  {size:,} bytes")
    else:
        print(f"{label:<25}  MISSING")